In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_hr_kpis AS

WITH employees AS (

    SELECT
        COUNT(DISTINCT e.employee_id)
            AS total_employees

    FROM gold.dim_employee e

    WHERE is_active = true

),

new_employees AS (

    SELECT
        COUNT(DISTINCT employee_id)
            AS new_employees_this_month

    FROM gold.dim_employee

    WHERE is_active = true
      AND YEAR(created_at) = YEAR(CURRENT_DATE)
      AND MONTH(created_at) = MONTH(CURRENT_DATE)

),

departments AS (

    SELECT
        COUNT(DISTINCT department_id)
            AS total_departments

    FROM gold.dim_department

    WHERE is_active = true

),

active_jobs AS (

    SELECT
        COUNT(DISTINCT job_id)
            AS active_jobs

    FROM gold.dim_job

    WHERE is_active = true

)

SELECT *
FROM employees
CROSS JOIN new_employees
CROSS JOIN departments
CROSS JOIN active_jobs;

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_employees_by_department AS

SELECT
    dd.department_name,

    COUNT(DISTINCT fe.employee_id)
        AS total_employees

FROM gold.fact_employee_snapshot fe

INNER JOIN gold.dim_department dd
    ON fe.department_id = dd.department_id

WHERE fe.is_active = true
  AND dd.is_active = true

GROUP BY
    dd.department_name;

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_employee_growth_trend AS

SELECT
    YEAR(snapshot_date) AS year_number,

    MONTH(snapshot_date) AS month_number,

    DATE_FORMAT(snapshot_date, 'MMM')
        AS month_short,

    COUNT(DISTINCT employee_id)
        AS total_employees

FROM gold.fact_employee_snapshot

WHERE is_active = true

GROUP BY
    YEAR(snapshot_date),
    MONTH(snapshot_date),
    DATE_FORMAT(snapshot_date, 'MMM')

ORDER BY
    year_number,
    month_number;

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_employee_list AS

WITH latest_snapshot AS (

    SELECT
        MAX(snapshot_date) AS latest_date

    FROM gold.fact_employee_snapshot

)

SELECT
    CONCAT('EMP-', fes.employee_id)
        AS employee_code,

    fes.employee_name,

    fes.department_name,

    fes.job_name AS position,

    CASE
        WHEN fes.is_active = true
            THEN 'Active'
        ELSE 'Inactive'
    END AS employee_status,

    CAST(de.created_at AS DATE)
        AS join_date

FROM gold.fact_employee_snapshot fes

INNER JOIN gold.dim_employee de
    ON fes.employee_id = de.employee_id

CROSS JOIN latest_snapshot ls

WHERE fes.snapshot_date = ls.latest_date

order by fes.employee_id desc;

In [0]:
Select * from smart_odoo.gold.vw_hr_kpis;
---------------------------------------------
Select * from smart_odoo.gold.vw_employees_by_department;
---------------------------------------------
Select * from smart_odoo.gold.vw_employee_growth_trend;
---------------------------------------------
Select * from smart_odoo.gold.vw_employee_list;